# Exploratory Data Analysis -- StudentLife Dataset
**MSc Data Science and AI | Visual Analytics for Digital Health**  
**Rohith Elanchezhian | Newcastle University | Supervisor: Alaa Alahmadi**

---

## Overview

This notebook is the second stage of the pipeline, coming after data cleaning.
The goal here is to understand the data before building any machine learning models.
I want to know what the distributions look like, whether there are any patterns across the 10-week term,
and whether any variables are correlated with stress -- the main outcome I am trying to predict.

I work through each variable systematically:

1. Response rate audit -- did students actually complete surveys consistently?
2. Stress analysis -- distribution, weekly trends, time of day, weekday vs weekend
3. Sleep analysis -- hours, quality, and how it relates to stress
4. Mood analysis -- daily mood patterns
5. Exercise and social behaviour
6. Passive sensor data -- physical activity, conversations, Bluetooth proximity
7. Temporal patterns -- all key variables on one chart
8. Per-student variability -- individual differences hidden by group averages
9. Correlation analysis -- which variables move together?
10. Statistical tests -- confirming key findings with t-tests and ANOVA

All charts are saved to Google Drive so they can be used in the dissertation and poster.

## Step 1 -- Mount Google Drive and Import Libraries

I start by mounting Drive and importing everything I need.
I also define a consistent colour palette so that all charts in this project
use the same colours for the same variables -- stress is always red, sleep always teal, and so on.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)

# Consistent colour palette -- same variable always same colour across all charts
PALETTE = {
    'stress':   '#E05C5C',
    'sleep':    '#4ECDC4',
    'mood':     '#7C6AF7',
    'social':   '#F0A050',
    'activity': '#50C878',
    'neutral':  '#8888AA',
}

# Clean chart style throughout
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor']   = '#FAFAFA'
plt.rcParams['axes.grid']        = True
plt.rcParams['grid.alpha']       = 0.3
plt.rcParams['axes.spines.top']  = False
plt.rcParams['axes.spines.right']= False

WEEKS     = list(range(1, 11))
DAY_NAMES = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

print('Libraries imported and chart style configured.')

## Step 2 -- Load the Clean CSV Files

I load all 7 clean CSV files produced by the data cleaning notebook.
The most important is `daily_master.csv` -- one row per student per day with all variables merged.
That is the main table used for correlation analysis and machine learning.

I also define a `save_fig()` helper so every chart automatically saves to Drive.

In [ ]:
# Change this if your clean_data folder is in a different location
CLEAN_DIR = '/content/drive/MyDrive/clean_data'

mood_df     = pd.read_csv(f'{CLEAN_DIR}/mood_clean.csv')
stress_df   = pd.read_csv(f'{CLEAN_DIR}/stress_clean.csv')
sleep_df    = pd.read_csv(f'{CLEAN_DIR}/sleep_clean.csv')
conv_df     = pd.read_csv(f'{CLEAN_DIR}/conversation_clean.csv')
bt_df       = pd.read_csv(f'{CLEAN_DIR}/bluetooth_clean.csv')
activity_df = pd.read_csv(f'{CLEAN_DIR}/activity_clean.csv')
master_df   = pd.read_csv(f'{CLEAN_DIR}/daily_master.csv')

# Convert date columns to proper datetime objects
for df in [mood_df, stress_df, sleep_df, conv_df, bt_df, activity_df, master_df]:
    if 'datetime' in df.columns:
        df['datetime'] = pd.to_datetime(df['datetime'], utc=True)
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])
for col in ['start_dt', 'end_dt']:
    if col in conv_df.columns:
        conv_df[col] = pd.to_datetime(conv_df[col], utc=True)

# Standardise student ID column name to 'uid'
for df in [mood_df, stress_df, sleep_df, conv_df, bt_df, activity_df, master_df]:
    if 'student_id' in df.columns:
        df.rename(columns={'student_id': 'uid'}, inplace=True)

# Add weekend flag where needed
for df in [mood_df, stress_df, sleep_df]:
    if 'day_of_week' in df.columns and 'is_weekend' not in df.columns:
        df['is_weekend'] = df['day_of_week'] >= 5

# Helper: saves chart to Drive and shows the filename
def save_fig(name):
    path = f'{CLEAN_DIR}/eda_{name}.png'
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print(f'  Saved: eda_{name}.png')

print('All files loaded successfully.')
print()
print(f'{"File":<22} {"Rows":>8}  {"Students":>10}')
print('-' * 45)
for name, df in [('mood_clean', mood_df), ('stress_clean', stress_df),
                  ('sleep_clean', sleep_df), ('conversation', conv_df),
                  ('bluetooth', bt_df), ('activity', activity_df),
                  ('daily_master', master_df)]:
    uids = df['uid'].nunique() if 'uid' in df.columns else '--'
    print(f'  {name:<20} {len(df):>8,}  {uids:>10}')

## Step 3 -- Response Rate Audit

Before diving into the data, I want to know how consistently students actually completed the surveys.
If most students stopped responding after week 3, all my later findings would be based on a small biased sample.
The heatmap below shows the number of stress responses per student per week,
which gives a clear picture of engagement across the term.

In [ ]:
print(f'{"Dataset":<22} {"Rows":>10}  {"Students":>10}  {"Coverage":>10}')
print('-' * 58)
datasets = [
    ('Mood (all folders)', mood_df),
    ('Stress',             stress_df),
    ('Sleep',              sleep_df),
    ('Conversation',       conv_df),
    ('Bluetooth',          bt_df),
    ('Activity',           activity_df),
    ('Daily master',       master_df),
]
for name, df in datasets:
    n    = len(df)
    uids = df['uid'].nunique() if 'uid' in df.columns else '--'
    cov  = f'{uids/49*100:.0f}%' if isinstance(uids, int) else '--'
    print(f'  {name:<20} {n:>10,}  {uids:>10}  {cov:>10}')

In [ ]:
# Response heatmap: how many stress surveys did each student complete each week?
# Dark red = many responses, white = no responses that week
pivot = (stress_df.groupby(['uid', 'study_week'])
         .size()
         .unstack(fill_value=0)
         .reindex(columns=WEEKS, fill_value=0))

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(pivot, ax=ax, cmap='YlOrRd', annot=True, fmt='d',
            annot_kws={'size': 7}, linewidths=0.3, linecolor='white',
            cbar_kws={'label': 'Response count'})
ax.set_title('Stress Survey Responses per Student per Week\n'
             'White = no data, dark red = many responses',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Study Week')
ax.set_ylabel('Student ID')
plt.tight_layout()
save_fig('01_response_heatmap')
plt.show()

sparse = (stress_df.groupby('uid').size() < 5).sum()
print(f'Students with fewer than 5 total stress responses: {sparse}')
print('These students are flagged as sparse data in the dashboard.')

## Step 4 -- Stress Analysis

Stress is the main variable in this project. I look at it from four angles:
1. Overall distribution -- what is the typical stress level?
2. Weekly trends -- does stress change across the term?
3. Time of day -- are students more stressed in the morning or evening?
4. Weekday vs weekend -- a counterintuitive finding emerges here

In [ ]:
# Overall stress distribution
valid = stress_df['stress_level'].dropna()
total = len(valid)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: histogram with percentage labels
axes[0].hist(valid, bins=[0.5, 1.5, 2.5, 3.5, 4.5, 5.5],
             color=PALETTE['stress'], edgecolor='white', linewidth=0.8, rwidth=0.85)
axes[0].set_xticks([1, 2, 3, 4, 5])
axes[0].set_xticklabels(['1\nNot stressed', '2', '3\nModerate', '4', '5\nVery stressed'])
axes[0].set_title('Stress Level Distribution', fontweight='bold')
axes[0].set_ylabel('Number of responses')
for level in [1, 2, 3, 4, 5]:
    n = (valid == level).sum()
    axes[0].text(level, n + 5, f'{n/total*100:.0f}%',
                 ha='center', fontsize=9, color='#444')

# Right: pie chart showing three broad categories
low  = (valid <= 2).sum()
mid  = (valid == 3).sum()
high = (valid >= 4).sum()
axes[1].pie(
    [low, mid, high],
    labels=[f'Low (1-2)\n{low/total*100:.0f}%',
            f'Moderate (3)\n{mid/total*100:.0f}%',
            f'High (4-5)\n{high/total*100:.0f}%'],
    colors=['#4ECDC4', '#F0A050', '#E05C5C'],
    startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[1].set_title('Stress Categories', fontweight='bold')

plt.tight_layout()
save_fig('02_stress_distribution')
plt.show()

print(f'Total responses : {total:,}')
print(f'Mean stress     : {valid.mean():.3f} / 5')
print(f'Median stress   : {valid.median():.1f}')
print(f'High stress rate: {high/total*100:.1f}% (level 4 or 5)')

In [ ]:
# Stress by study week -- the key temporal pattern
weekly = (stress_df.groupby('study_week')['stress_level']
          .agg(['mean', 'sem', 'count']).reset_index())

fig, ax = plt.subplots(figsize=(11, 5))

# Highlight finals week (week 10) in full red; other weeks in lighter shade
bar_colors = [PALETTE['stress'] if w == 10 else '#CCAAAA'
              for w in weekly['study_week']]
ax.bar(weekly['study_week'], weekly['mean'],
       color=bar_colors, edgecolor='white', linewidth=0.5, zorder=2)
ax.errorbar(weekly['study_week'], weekly['mean'], yerr=weekly['sem'],
            fmt='none', color='#555', capsize=4, linewidth=1.5, zorder=3)

# Label each bar with the response count
for _, row in weekly.iterrows():
    ax.text(row['study_week'], row['mean'] + row['sem'] + 0.06,
            f'n={int(row["count"])}', ha='center', fontsize=8, color='#777')

ax.axhline(y=valid.mean(), color='#444', linestyle='--', linewidth=1,
           label=f'Overall mean ({valid.mean():.2f})')
ax.axvspan(9.5, 10.5, alpha=0.08, color='red')  # shade finals week
ax.text(10, 4.7, 'Finals', ha='center', color='red', fontsize=9)
ax.set_xticks(WEEKS)
ax.set_xticklabels([f'Week {w}' for w in WEEKS])
ax.set_ylim(1, 5)
ax.set_ylabel('Average stress level (1-5)', fontsize=11)
ax.set_title('Average Stress Level by Study Week  (error bars = +/-1 SE)',
             fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
save_fig('03_stress_by_week')
plt.show()

print('Stress by week:')
for _, r in weekly.iterrows():
    bar = chr(9608) * int(r['mean'] * 10)
    print(f'  Week {int(r["study_week"]):2d}: {r["mean"]:.2f}  {bar}  (n={int(r["count"])})')

In [ ]:
# Stress by hour of day -- when do students feel most stressed?
# This is only meaningful because we converted timestamps to EDT in the cleaning stage
hourly = stress_df.groupby('hour')['stress_level'].agg(['mean', 'count']).reset_index()

fig, ax = plt.subplots(figsize=(13, 4))
colors = [PALETTE['stress'] if h in [6, 7, 8, 9] else PALETTE['neutral']
          for h in hourly['hour']]
ax.bar(hourly['hour'], hourly['mean'], color=colors, edgecolor='white', linewidth=0.4)
for _, row in hourly.iterrows():
    ax.text(row['hour'], 1.02, f'n={int(row["count"])}',
            ha='center', fontsize=7, color='#AAA', rotation=90)
ax.axhline(y=valid.mean(), color='#444', linestyle='--', linewidth=1,
           alpha=0.7, label='Overall mean')
ax.set_xticks(range(24))
ax.set_xticklabels([f'{h}:00' for h in range(24)], rotation=45, ha='right', fontsize=8)
ax.set_ylim(1, 4)
ax.set_ylabel('Average stress level')
ax.set_title('Stress by Hour of Day (EDT)  |  Red = 6-9am morning peak',
             fontsize=12, fontweight='bold')
ax.legend()
ax.axvspan(-0.5, 5.5, alpha=0.05, color='navy')
ax.text(2.5, 3.7, 'Night', ha='center', color='#AAA', fontsize=9)
plt.tight_layout()
save_fig('04_stress_by_hour')
plt.show()

In [ ]:
# Weekday vs weekend stress -- I expected weekends to be lower
# The t-test below tells us whether the difference is statistically significant
wd = stress_df[stress_df['is_weekend'] == False]['stress_level'].dropna()
we = stress_df[stress_df['is_weekend'] == True]['stress_level'].dropna()
t, p = stats.ttest_ind(wd, we)

fig, ax = plt.subplots(figsize=(5, 4))
means = [wd.mean(), we.mean()]
sems  = [wd.sem(),  we.sem()]
bars  = ax.bar(['Weekday', 'Weekend'], means,
               color=[PALETTE['neutral'], PALETTE['social']],
               edgecolor='white', width=0.5)
ax.errorbar(['Weekday', 'Weekend'], means, yerr=sems,
            fmt='none', color='#555', capsize=6, linewidth=2)
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2,
            mean - 0.08, f'{mean:.2f}',
            ha='center', fontsize=12, fontweight='bold', color='white')

sig_label = 'significant p<.05' if p < .05 else f'not significant (p={p:.3f})'
ax.set_title(f'Weekday vs Weekend Stress\n{sig_label}', fontweight='bold')
ax.set_ylabel('Average stress level')
ax.set_ylim(1, 3.5)
plt.tight_layout()
save_fig('05_weekday_weekend')
plt.show()

print(f'Weekday mean: {wd.mean():.3f}  (n={len(wd):,})')
print(f'Weekend mean: {we.mean():.3f}  (n={len(we):,})')
print(f't = {t:.3f},  p = {p:.4f}  ({"SIGNIFICANT" if p<.05 else "not significant"})')
print()
if we.mean() > wd.mean():
    print('Weekend stress is HIGHER than weekday stress.')
    print('This is counterintuitive -- unstructured time may increase anxiety rather than relieve it.')

## Step 5 -- Sleep Analysis

Sleep is one of the strongest predictors of stress in this dataset.
I look at the distribution of sleep hours, the proportion of nights that fell below
the recommended 7 hours, and how sleep changed week by week across the term.

In [ ]:
valid_h = sleep_df['sleep_hours'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Sleep hours histogram with recommended range lines
axes[0].hist(valid_h, bins=range(0, 17),
             color=PALETTE['sleep'], edgecolor='white', linewidth=0.8, rwidth=0.9)
axes[0].axvline(x=7, color='green', linestyle='--', linewidth=2,
                label='7h minimum (recommended)')
axes[0].axvline(x=9, color='orange', linestyle='--', linewidth=2,
                label='9h maximum (recommended)')
axes[0].axvline(x=valid_h.mean(), color=PALETTE['stress'], linestyle=':',
                linewidth=2, label=f'Mean ({valid_h.mean():.1f}h)')
axes[0].set_xlabel('Hours of sleep')
axes[0].set_ylabel('Number of responses')
axes[0].set_title('Sleep Hours Distribution', fontweight='bold')
axes[0].legend(fontsize=9)

# Sleep categories bar chart
cats = pd.cut(valid_h,
              bins=[-np.inf, 5, 7, 9, np.inf],
              labels=['Under 5h\n(deprived)', '5-7h\n(short)',
                      '7-9h\n(adequate)', 'Over 9h\n(long)'])
cat_counts   = cats.value_counts().sort_index()
colors_cat   = ['#E05C5C', '#F0A050', '#4ECDC4', '#7C6AF7']
bars = axes[1].bar(cat_counts.index, cat_counts.values,
                   color=colors_cat, edgecolor='white')
for bar, v in zip(bars, cat_counts.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 3,
                 f'{v/len(valid_h)*100:.1f}%',
                 ha='center', fontsize=10, fontweight='bold')
axes[1].set_title('Sleep Duration Categories', fontweight='bold')
axes[1].set_ylabel('Number of nights')

plt.tight_layout()
save_fig('06_sleep_distribution')
plt.show()

print(f'Mean sleep       : {valid_h.mean():.2f} hours')
print(f'Median sleep     : {valid_h.median():.1f} hours')
print(f'Nights under 7h  : {(valid_h < 7).mean()*100:.1f}%')
print(f'Nights under 5h  : {(valid_h < 5).mean()*100:.1f}%  (deprived)')
print(f'Sleep quality    : {sleep_df["sleep_rate"].mean():.2f} / 5')

In [ ]:
# Sleep by study week -- does it drop during finals?
sleep_weekly = (sleep_df.groupby('study_week')['sleep_hours']
                .agg(['mean', 'sem', 'count']).reset_index())

fig, ax = plt.subplots(figsize=(11, 5))

# Colour-code bars: teal = adequate (7h+), orange = short (5-7h), red = deprived
bar_colors = ['#4ECDC4' if m >= 7 else '#F0A050' if m >= 5 else '#E05C5C'
              for m in sleep_weekly['mean']]
ax.bar(sleep_weekly['study_week'], sleep_weekly['mean'],
       color=bar_colors, edgecolor='white', linewidth=0.5)
ax.errorbar(sleep_weekly['study_week'], sleep_weekly['mean'],
            yerr=sleep_weekly['sem'], fmt='none', color='#555', capsize=4)
for _, row in sleep_weekly.iterrows():
    ax.text(row['study_week'], row['mean'] + row['sem'] + 0.1,
            f'n={int(row["count"])}', ha='center', fontsize=8, color='#777')

ax.axhline(y=7, color='green', linestyle='--', linewidth=1.5,
           label='7h minimum recommended')
ax.set_xticks(WEEKS)
ax.set_xticklabels([f'Week {w}' for w in WEEKS])
ax.set_ylim(0, 12)
ax.set_ylabel('Average sleep hours', fontsize=11)
ax.set_title('Sleep Duration by Study Week  (teal = adequate, orange = short, red = deprived)',
             fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
save_fig('07_sleep_by_week')
plt.show()

w10_sleep = sleep_df[sleep_df['study_week'] == 10]['sleep_hours'].mean()
w78_sleep = sleep_df[sleep_df['study_week'].isin([7, 8])]['sleep_hours'].mean()
print(f'Finals week (10) mean sleep : {w10_sleep:.2f}h  -- lowest of the term')
print(f'Weeks 7-8 mean sleep        : {w78_sleep:.2f}h  -- highest (reading period)')

## Step 6 -- Mood Analysis

The mood score is computed as happy minus sad (range -4 to +4).
A score above zero means the student felt happier than sad.
I look at the distribution across the term and by day of week.

In [ ]:
mood_valid = mood_df.dropna(subset=['mood_score'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution of mood scores
axes[0].hist(mood_valid['mood_score'], bins=range(-4, 6),
             color=PALETTE['mood'], edgecolor='white', rwidth=0.85)
axes[0].axvline(x=0, color='black', linestyle='--', linewidth=1.5,
                label='Neutral (0)')
axes[0].axvline(x=mood_valid['mood_score'].mean(), color=PALETTE['stress'],
                linestyle=':', linewidth=2,
                label=f'Mean ({mood_valid["mood_score"].mean():.2f})')
axes[0].set_xlabel('Mood score (happy - sad)')
axes[0].set_ylabel('Responses')
axes[0].set_title('Mood Score Distribution', fontweight='bold')
axes[0].legend()

# Mood by day of week
if 'day_of_week' in mood_valid.columns:
    day_mood = mood_valid.groupby('day_of_week')['mood_score'].mean()
    colors   = [PALETTE['social'] if i >= 5 else PALETTE['mood'] for i in range(7)]
    axes[1].bar(range(7), day_mood.values, color=colors, edgecolor='white')
    axes[1].set_xticks(range(7))
    axes[1].set_xticklabels(DAY_NAMES)
    axes[1].axhline(y=0, color='black', linestyle='--', linewidth=1)
    axes[1].set_ylabel('Average mood score')
    axes[1].set_title('Mood Score by Day of Week\n(purple = weekday, orange = weekend)',
                      fontweight='bold')

plt.tight_layout()
save_fig('08_mood')
plt.show()

print(f'Mean mood score : {mood_valid["mood_score"].mean():.3f}')
print(f'Positive days   : {(mood_valid["mood_score"] > 0).mean()*100:.1f}%')
print(f'Negative days   : {(mood_valid["mood_score"] < 0).mean()*100:.1f}%')

## Step 7 -- Exercise and Social Behaviour

Exercise is one of the most commonly recommended interventions for stress and mood.
I want to know how often students actually exercised during the study, and whether exercise days
had measurably lower stress levels.

The social analysis looks at how many people students interacted with per day
and whether social contact is associated with better mood.

In [ ]:
# Exercise analysis
if 'exercised' in master_df.columns and 'stress_avg' in master_df.columns:

    ex_rate_by_week = master_df.groupby('study_week')['exercised_today'].apply(
        lambda x: pd.to_numeric(x.map({'True': 1, 'False': 0, True: 1, False: 0}),
                                errors='coerce').mean()
    )

    # Stress on exercise vs non-exercise days using master table
    mast = master_df.copy()
    mast['ex_bool'] = mast.get('exercised_today', mast.get('exercised',
                               pd.Series())).map(
        {'True': True, 'False': False, True: True, False: False})

    ex_yes  = mast[mast['ex_bool'] == True]['stress_avg'].dropna()
    ex_no   = mast[mast['ex_bool'] == False]['stress_avg'].dropna()
    t_ex, p_ex = stats.ttest_ind(ex_yes, ex_no)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Exercise rate by week
    axes[0].bar(ex_rate_by_week.index, ex_rate_by_week.values * 100,
                color=PALETTE['activity'], edgecolor='white')
    axes[0].set_xticks(WEEKS)
    axes[0].set_xticklabels([f'W{w}' for w in WEEKS])
    axes[0].set_ylabel('% of days with exercise')
    axes[0].set_title('Exercise Rate by Study Week', fontweight='bold')

    # Stress comparison
    bars = axes[1].bar(['Exercised', 'Did not exercise'],
                       [ex_yes.mean(), ex_no.mean()],
                       color=[PALETTE['activity'], PALETTE['neutral']],
                       edgecolor='white', width=0.5)
    for bar, m in zip(bars, [ex_yes.mean(), ex_no.mean()]):
        axes[1].text(bar.get_x() + bar.get_width() / 2, m - 0.08,
                     f'{m:.2f}', ha='center', fontsize=12, fontweight='bold', color='white')
    sig = 'p<.05 significant' if p_ex < .05 else f'p={p_ex:.3f}'
    axes[1].set_title(f'Stress on Exercise vs Non-exercise Days\n{sig}', fontweight='bold')
    axes[1].set_ylabel('Mean stress level')
    axes[1].set_ylim(1, 3.5)

    plt.tight_layout()
    save_fig('09_exercise')
    plt.show()

    overall_ex_rate = ex_rate_by_week.mean() * 100
    print(f'Overall exercise rate : {overall_ex_rate:.1f}% of days')
    print(f'Stress (exercised)    : {ex_yes.mean():.3f}')
    print(f'Stress (no exercise)  : {ex_no.mean():.3f}')
    print(f't = {t_ex:.3f},  p = {p_ex:.4f}')
else:
    print('Exercise columns not found in master -- check column names.')

## Step 8 -- Passive Sensing: Activity, Conversation, Bluetooth

These three sensor streams capture physical and social behaviour without requiring
any active input from students. I look at the daily patterns for each one.

In [ ]:
# Physical activity patterns by hour of day
if 'activity_code' in activity_df.columns and 'hour' in activity_df.columns:
    act_hourly = (
        activity_df.groupby('hour')
        .apply(lambda x: pd.Series({
            'stationary': (x['activity_code'] == 0).mean() * 100,
            'walking':    (x['activity_code'] == 1).mean() * 100,
            'running':    (x['activity_code'] == 2).mean() * 100,
        }))
    ).reset_index()

    fig, ax = plt.subplots(figsize=(13, 4))
    ax.fill_between(act_hourly['hour'], act_hourly['stationary'],
                    label='Stationary', color=PALETTE['neutral'], alpha=0.6)
    ax.fill_between(act_hourly['hour'], act_hourly['walking'],
                    label='Walking', color=PALETTE['sleep'], alpha=0.8)
    ax.fill_between(act_hourly['hour'], act_hourly['running'],
                    label='Running', color=PALETTE['activity'], alpha=0.8)
    ax.set_xticks(range(24))
    ax.set_xticklabels([f'{h}:00' for h in range(24)], rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('% of sensor readings')
    ax.set_title('Physical Activity Patterns by Hour of Day', fontsize=12, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    save_fig('10_activity_hourly')
    plt.show()

    total_readings  = len(activity_df)
    stationary_pct  = (activity_df['activity_code'] == 0).mean() * 100
    walking_pct     = (activity_df['activity_code'] == 1).mean() * 100
    running_pct     = (activity_df['activity_code'] == 2).mean() * 100
    print(f'Total activity readings : {total_readings:,}')
    print(f'Stationary : {stationary_pct:.1f}%')
    print(f'Walking    : {walking_pct:.1f}%')
    print(f'Running    : {running_pct:.1f}%')

In [ ]:
# Conversation patterns -- when and how long do students talk?
if 'duration_min' in conv_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Duration distribution
    axes[0].hist(conv_df['duration_min'].clip(0, 60), bins=30,
                 color=PALETTE['social'], edgecolor='white')
    axes[0].set_xlabel('Conversation duration (minutes)')
    axes[0].set_ylabel('Number of events')
    axes[0].set_title('Conversation Duration Distribution\n(clipped at 60 min for readability)',
                      fontweight='bold')

    # Total talking time by hour
    if 'hour' in conv_df.columns:
        hourly_talk = conv_df.groupby('hour')['duration_min'].sum()
        axes[1].bar(hourly_talk.index, hourly_talk.values,
                    color=PALETTE['social'], edgecolor='white')
        axes[1].set_xlabel('Hour of day (EDT)')
        axes[1].set_ylabel('Total talking time (minutes)')
        axes[1].set_title('Total Conversation Time by Hour', fontweight='bold')

    plt.tight_layout()
    save_fig('11_conversation')
    plt.show()

    print(f'Total conversation events : {len(conv_df):,}')
    print(f'Mean duration             : {conv_df["duration_min"].mean():.1f} minutes')
    print(f'Median duration           : {conv_df["duration_min"].median():.1f} minutes')

## Step 9 -- Temporal Patterns: All Key Variables Together

To get a full picture of how the term progressed, I plot all four key variables
(stress, sleep, mood, and talking time) on a single four-panel chart.
This makes it easy to see whether different variables move together
and whether the finals week pattern appears across all of them.

In [ ]:
# Four-panel weekly trends chart using the daily master table
if 'stress_avg' in master_df.columns:
    weekly_master = master_df.groupby('study_week').agg(
        stress  = ('stress_avg',            'mean'),
        sleep   = ('sleep_hours',           'mean'),
        mood    = ('mood_score_avg',         'mean'),
        talking = ('total_talking_minutes',  'mean'),
    ).reindex(WEEKS)

    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    fig.suptitle('Key Variables Across the 10-Week Academic Term',
                 fontsize=14, fontweight='bold')

    plots = [
        (axes[0, 0], 'stress',  PALETTE['stress'], 'Stress Level (1-5)', [1, 5]),
        (axes[0, 1], 'sleep',   PALETTE['sleep'],  'Sleep Hours',        [3, 12]),
        (axes[1, 0], 'mood',    PALETTE['mood'],   'Mood Score',         [-2, 2.5]),
        (axes[1, 1], 'talking', PALETTE['social'], 'Talking Time (min)', [0, 70]),
    ]

    for ax, col, color, ylabel, ylim in plots:
        y = weekly_master[col].dropna()
        ax.plot(y.index, y.values, 'o-', color=color, linewidth=2.5,
                markersize=8, markerfacecolor='white', markeredgewidth=2.5)
        ax.fill_between(y.index, y.values, alpha=0.15, color=color)
        ax.axvspan(9.5, 10.5, alpha=0.08, color='red')
        ax.set_xticks(WEEKS)
        ax.set_xticklabels([f'W{w}' for w in WEEKS], fontsize=8)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_ylim(ylim)

    plt.tight_layout()
    save_fig('12_weekly_trends')
    plt.show()
else:
    print('stress_avg column not found in master_df -- run the cleaning notebook first.')

## Step 10 -- Per-Student Variability

Group averages can hide enormous individual differences.
This chart shows the average stress level for each of the 49 students,
sorted from lowest to highest.
The range reveals how much students vary -- the most stressed student has an average
almost twice as high as the least stressed.

In [ ]:
per_student = (stress_df.groupby('uid')['stress_level']
               .agg(['mean', 'std', 'count'])
               .reset_index()
               .sort_values('mean'))

fig, ax = plt.subplots(figsize=(14, 5))
colors = [PALETTE['stress'] if m >= 3.5
          else PALETTE['social'] if m >= 2.5
          else PALETTE['sleep']
          for m in per_student['mean']]

ax.bar(range(len(per_student)), per_student['mean'],
       color=colors, edgecolor='white', linewidth=0.4)
ax.errorbar(range(len(per_student)), per_student['mean'],
            yerr=per_student['std'] / np.sqrt(per_student['count']),
            fmt='none', color='#555', capsize=2, linewidth=0.8)
ax.axhline(y=stress_df['stress_level'].mean(), color='black',
           linestyle='--', linewidth=1, label='Group mean')
ax.set_xticks(range(len(per_student)))
ax.set_xticklabels(per_student['uid'].tolist(), rotation=90, fontsize=7)
ax.set_ylabel('Average stress level', fontsize=11)
ax.set_title('Average Stress Level per Student (sorted low to high)\n'
             'Teal = low  |  Orange = moderate  |  Red = high',
             fontsize=12, fontweight='bold')
ax.legend()
ax.set_ylim(1, 5)
plt.tight_layout()
save_fig('13_per_student')
plt.show()

print(f'Most stressed student  : {per_student.iloc[-1]["uid"]}  (mean = {per_student.iloc[-1]["mean"]:.2f})')
print(f'Least stressed student : {per_student.iloc[0]["uid"]}  (mean = {per_student.iloc[0]["mean"]:.2f})')
print(f'Range                  : {per_student["mean"].max() - per_student["mean"].min():.2f} stress units')

## Step 11 -- Correlation Analysis

Now I look at which variables are correlated with each other using the daily master table.
I compute the Pearson correlation coefficient between all pairs of key variables
and visualise the result as a heatmap.
I also plot scatter diagrams for the most interesting relationships.

In [ ]:
# Correlation matrix of key variables from the daily master table
corr_cols = {
    'stress_avg':           'Stress',
    'sleep_hours':          'Sleep hrs',
    'sleep_rate':           'Sleep quality',
    'mood_score_avg':       'Mood',
    'fraction_walking':     'Walking',
    'total_talking_minutes':'Talking',
    'unique_devices_nearby':'BT devices',
}

# Only use columns that actually exist in this file
available = {k: v for k, v in corr_cols.items() if k in master_df.columns}

if len(available) >= 3:
    corr_data = master_df[list(available.keys())].rename(columns=available)
    corr_mat  = corr_data.corr()

    # Mask the upper triangle so we do not show duplicate values
    mask = np.triu(np.ones(corr_mat.shape, dtype=bool), k=1)

    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(corr_mat, mask=mask, ax=ax,
                annot=True, fmt='.2f', cmap='RdBu_r',
                vmin=-1, vmax=1, linewidths=0.5,
                cbar_kws={'label': 'Pearson r'})
    ax.set_title('Correlation Matrix -- Key Daily Variables',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    save_fig('14_correlation_matrix')
    plt.show()

    # Print the strongest correlations with stress
    print('Correlations with stress (strongest first):')
    stress_corrs = corr_mat['Stress'].drop('Stress').sort_values(key=abs, ascending=False)
    for var, r in stress_corrs.items():
        direction = 'positive' if r > 0 else 'negative'
        print(f'  {var:<20} r = {r:.3f}  ({direction})')
else:
    print('Not enough columns available for correlation analysis.')

In [ ]:
# Scatter plots for the two most interesting relationships
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

if 'sleep_hours' in master_df.columns and 'stress_avg' in master_df.columns:
    d = master_df[['sleep_hours', 'stress_avg']].dropna()
    r, p = stats.pearsonr(d['sleep_hours'], d['stress_avg'])
    m, b = np.polyfit(d['sleep_hours'], d['stress_avg'], 1)
    xs   = np.linspace(d['sleep_hours'].min(), d['sleep_hours'].max(), 100)
    axes[0].scatter(d['sleep_hours'], d['stress_avg'],
                    alpha=0.3, color=PALETTE['stress'], s=20)
    axes[0].plot(xs, m * xs + b, 'k--', linewidth=2)
    axes[0].set_xlabel('Sleep hours')
    axes[0].set_ylabel('Stress level')
    axes[0].set_title(f'Sleep vs Stress  (r = {r:.3f}, p = {p:.4f})', fontweight='bold')

if 'total_talking_minutes' in master_df.columns and 'mood_score_avg' in master_df.columns:
    d2 = master_df[['total_talking_minutes', 'mood_score_avg']].dropna()
    r2, p2 = stats.pearsonr(d2['total_talking_minutes'], d2['mood_score_avg'])
    m2, b2 = np.polyfit(d2['total_talking_minutes'], d2['mood_score_avg'], 1)
    xs2    = np.linspace(d2['total_talking_minutes'].min(), d2['total_talking_minutes'].max(), 100)
    axes[1].scatter(d2['total_talking_minutes'], d2['mood_score_avg'],
                    alpha=0.3, color=PALETTE['social'], s=20)
    axes[1].plot(xs2, m2 * xs2 + b2, 'k--', linewidth=2)
    axes[1].set_xlabel('Total talking time (min)')
    axes[1].set_ylabel('Mood score')
    axes[1].set_title(f'Talking vs Mood  (r = {r2:.3f}, p = {p2:.4f})', fontweight='bold')

plt.tight_layout()
save_fig('15_scatter_plots')
plt.show()

## Step 12 -- Statistical Tests

I use five statistical tests to confirm whether the patterns I observed visually are
statistically significant -- i.e. unlikely to have occurred by chance.

- **ANOVA** tests whether stress differs significantly across all 10 weeks
- **t-test 1** compares finals week stress to the rest of term
- **t-test 2** compares weekday vs weekend stress (the counterintuitive finding)
- **t-test 3** compares stress on days with and without adequate sleep (7h+)
- **t-test 4** compares stress on exercise vs non-exercise days

For each test, a p-value below 0.05 means the result is statistically significant.

In [ ]:
print('=' * 62)
print('TEST 1 -- ANOVA: does stress differ across the 10 weeks?')
print('=' * 62)
groups_by_week = [stress_df[stress_df['study_week'] == w]['stress_level'].dropna()
                  for w in WEEKS]
f, p = stats.f_oneway(*groups_by_week)
print(f'F = {f:.3f},  p = {p:.4f}  -- {"SIGNIFICANT" if p<.05 else "not significant"}')
print()

print('=' * 62)
print('TEST 2 -- t-test: finals week (W10) vs rest of term')
print('=' * 62)
finals  = stress_df[stress_df['study_week'] == 10]['stress_level'].dropna()
rest    = stress_df[stress_df['study_week'] != 10]['stress_level'].dropna()
t2, p2  = stats.ttest_ind(finals, rest)
d2      = (finals.mean() - rest.mean()) / rest.std()  # Cohen's d
print(f'Finals mean : {finals.mean():.3f}')
print(f'Rest mean   : {rest.mean():.3f}')
print(f't = {t2:.3f},  p = {p2:.4f},  Cohen d = {d2:.3f}  -- {"SIGNIFICANT" if p2<.05 else "not significant"}')
print()

print('=' * 62)
print('TEST 3 -- t-test: weekday vs weekend stress')
print('=' * 62)
wd3 = stress_df[stress_df['is_weekend'] == False]['stress_level'].dropna()
we3 = stress_df[stress_df['is_weekend'] == True]['stress_level'].dropna()
t3, p3 = stats.ttest_ind(wd3, we3)
print(f'Weekday mean : {wd3.mean():.3f}')
print(f'Weekend mean : {we3.mean():.3f}')
print(f't = {t3:.3f},  p = {p3:.4f}  -- {"SIGNIFICANT" if p3<.05 else "not significant"}')
print()

print('=' * 62)
print('TEST 4 -- t-test: sleep-deprived (<7h) vs adequate (>=7h)')
print('=' * 62)
if 'sleep_hours' in master_df.columns and 'stress_avg' in master_df.columns:
    deprived = master_df[master_df['sleep_hours'] < 7]['stress_avg'].dropna()
    adequate = master_df[master_df['sleep_hours'] >= 7]['stress_avg'].dropna()
    t4, p4 = stats.ttest_ind(deprived, adequate)
    print(f'Deprived (<7h) mean  : {deprived.mean():.3f}  (n={len(deprived):,})')
    print(f'Adequate (>=7h) mean : {adequate.mean():.3f}  (n={len(adequate):,})')
    print(f't = {t4:.3f},  p = {p4:.4f}  -- {"SIGNIFICANT" if p4<.05 else "not significant"}')
    print()

print('=' * 62)
print('TEST 5 -- t-test: exercise vs no exercise days')
print('=' * 62)
if 'exercised_today' in master_df.columns and 'stress_avg' in master_df.columns:
    mdf = master_df.copy()
    mdf['ex'] = mdf['exercised_today'].map(
        {'True': True, 'False': False, True: True, False: False})
    ex_y = mdf[mdf['ex'] == True]['stress_avg'].dropna()
    ex_n = mdf[mdf['ex'] == False]['stress_avg'].dropna()
    t5, p5 = stats.ttest_ind(ex_y, ex_n)
    print(f'Exercised mean     : {ex_y.mean():.3f}  (n={len(ex_y):,})')
    print(f'No exercise mean   : {ex_n.mean():.3f}  (n={len(ex_n):,})')
    print(f't = {t5:.3f},  p = {p5:.4f}  -- {"SIGNIFICANT" if p5<.05 else "not significant"}')

## Step 13 -- Key Findings Summary

Here is a concise summary of the most important findings from the EDA.
These findings directly motivate the machine learning models in the next notebook.

In [ ]:
print('=' * 66)
print('EDA KEY FINDINGS -- StudentLife Dataset')
print('=' * 66)
print()
print('STRESS')
print(f'  Mean stress level : {stress_df["stress_level"].mean():.2f} / 5')
print(f'  High stress rate  : {(stress_df["stress_level"] >= 4).mean()*100:.1f}% (level 4-5)')
print(f'  Finals week mean  : {finals.mean():.2f}  vs term average {rest.mean():.2f}  (p={p2:.3f})')
print(f'  Weekend mean      : {we3.mean():.2f}  vs weekday {wd3.mean():.2f}  (p={p3:.3f})')
print()
print('SLEEP')
print(f'  Mean sleep        : {sleep_df["sleep_hours"].mean():.2f}h per night')
print(f'  Nights under 7h   : {(sleep_df["sleep_hours"] < 7).mean()*100:.1f}%')
print(f'  Sleep quality     : {sleep_df["sleep_rate"].mean():.2f} / 5')
print(f'  Finals week sleep : {w10_sleep:.2f}h  (lowest of term)')
print()
print('KEY STATISTICAL TESTS')
print(f'  ANOVA (stress across weeks) : F={f:.2f}, p={p:.4f}')
print(f'  Finals vs rest of term      : t={t2:.2f}, p={p2:.4f}, d={d2:.2f}')
print(f'  Weekend vs weekday          : t={t3:.2f}, p={p3:.4f}')
print()
print('All charts saved to Google Drive as eda_*.png files.')